# Implementation Figures: Slides 2 To 5

This notebook builds PowerPoint-ready static assets for the first implementation-phase slides. Each figure is exported to `outputs/implementation_phase_python/` so it can be pasted into PowerPoint and combined with separate text boxes as needed.

In [25]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 120)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

In [26]:
ROOT = Path.cwd().resolve()
# Notebooks live in <project_root>/notebooks/ - navigate up to project root
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

PROCESSED_DIR = ROOT / 'data' / 'processed'
OUTPUT_DIR = ROOT / 'outputs' / 'implementation_phase_python'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

q1_2025 = pd.read_csv(PROCESSED_DIR / 'q1_2025_base_dref.csv', parse_dates=['approval_date'])
q1_2026 = pd.read_csv(PROCESSED_DIR / 'q1_2026_base_dref.csv', parse_dates=['approval_date'])
q1_history = pd.read_csv(PROCESSED_DIR / 'q1_history_base_dref.csv', parse_dates=['approval_date'])
protocol_context = pd.read_csv(PROCESSED_DIR / 'q1_2026_protocol_context.csv', parse_dates=['approval_date'])
metric_contract = pd.read_csv(PROCESSED_DIR / 'metric_contract.csv')

# ═════════════════════════════════════════════════════════════════════════════
#  Flourish-inspired style system (shared with all notebooks)
# ═════════════════════════════════════════════════════════════════════════════

# Backgrounds & grid
FLR_PAPER = '#fafafa'
FLR_PLOT = '#fafafa'
FLR_GRID = '#e8e8e8'
FLR_LINE = '#d0d0d0'
FLR_MUTED = '#8a8f98'

# Typography
FLR_FONT = 'Inter, Segoe UI, Helvetica Neue, Arial, sans-serif'
FLR_TITLE = {'size': 29, 'color': '#1a1a2e', 'family': f'Inter, Segoe UI Semibold, {FLR_FONT}'}
FLR_SUBTITLE = {'size': 17, 'color': FLR_MUTED, 'family': FLR_FONT}
FLR_AXIS_LBL = {'size': 18, 'color': '#333333', 'family': FLR_FONT}
FLR_TICK = {'size': 16, 'color': '#555555', 'family': FLR_FONT}
FLR_NOTE = {'size': 15, 'color': '#555555', 'family': FLR_FONT}
FLR_LEGEND = {'size': 15, 'color': '#555555', 'family': FLR_FONT}

# Colours
IFRC_RED = '#E03C31'
NAVY = '#2B3A67'
AMBER = '#F5A623'
SOFT_AMBER = '#f6d7a8'
WARM_ORANGE = '#F08A4B'
GREY = '#D5D5D5'
INK = '#222222'
BG = '#fafafa'


def flr_legend(y=-0.16, x=0.5):
    return dict(
        orientation='h',
        xanchor='center',
        x=x,
        yanchor='top',
        y=y,
        bgcolor='rgba(0,0,0,0)',
        borderwidth=0,
        font=FLR_LEGEND,
        itemsizing='constant',
    )


def flr_xaxis(title='', **overrides):
    base = dict(
        title={'text': title, 'font': FLR_AXIS_LBL, 'standoff': 18},
        tickfont=FLR_TICK,
        ticks='outside',
        ticklen=6,
        tickwidth=1,
        tickcolor=FLR_LINE,
        showgrid=False,
        zeroline=False,
        showline=True,
        linecolor=FLR_LINE,
        linewidth=0.9,
        automargin=True,
    )
    base.update(overrides)
    return base


def flr_yaxis(title='', **overrides):
    base = dict(
        title={'text': title, 'font': FLR_AXIS_LBL, 'standoff': 18},
        tickfont=FLR_TICK,
        ticks='outside',
        ticklen=6,
        tickwidth=1,
        tickcolor=FLR_LINE,
        gridcolor=FLR_GRID,
        gridwidth=0.6,
        showgrid=True,
        zeroline=False,
        showline=False,
        automargin=True,
    )
    base.update(overrides)
    return base


def flr_layout(title_text, subtitle='', height=700,
               margin_l=80, margin_r=40, margin_t=140, margin_b=120):
    title_html = title_text
    if subtitle:
        title_html += f'<br><span style="font-size:17px;color:{FLR_MUTED};">{subtitle}</span>'
    return dict(
        title={
            'text': title_html,
            'x': 0.5,
            'xanchor': 'center',
            'y': 0.98,
            'yanchor': 'top',
            'font': FLR_TITLE,
            'pad': {'t': 8, 'b': 26},
        },
        paper_bgcolor=FLR_PAPER,
        plot_bgcolor=FLR_PLOT,
        margin={'l': margin_l, 'r': margin_r, 't': margin_t, 'b': margin_b},
        height=height,
        font={'family': FLR_FONT, 'size': 15, 'color': '#444444'},
        hoverlabel=dict(
            bgcolor='white',
            bordercolor='#ddd',
            font_size=13,
            font_family=FLR_FONT,
        ),
    )


def human_chf(value: float) -> str:
    if pd.isna(value):
        return 'n/a'
    if abs(value) >= 1_000_000:
        return f'CHF {value / 1_000_000:.1f}M'
    if abs(value) >= 1_000:
        return f'CHF {value / 1_000:.0f}K'
    return f'CHF {value:,.0f}'


def human_number(value: float) -> str:
    if pd.isna(value):
        return 'n/a'
    if abs(value) >= 1_000_000:
        return f'{value / 1_000_000:.1f}M'
    if abs(value) >= 1_000:
        return f'{value / 1_000:.0f}K'
    return f'{value:,.0f}'


def save_figure(fig: go.Figure, stem: str, width: int = 1400, height: int = 900) -> None:
    fig.write_image(OUTPUT_DIR / f'{stem}.png', width=width, height=height, scale=2)
    fig.write_image(OUTPUT_DIR / f'{stem}.svg', width=width, height=height)


print('Processed dir:', PROCESSED_DIR)
print('Output dir:', OUTPUT_DIR)

Processed dir: C:\Users\arun.gandhi\Downloads\DREF_GA_visualizations\data\processed
Output dir: C:\Users\arun.gandhi\Downloads\DREF_GA_visualizations\outputs\implementation_phase_python


## Slide 2

This slide keeps KPI cards for fast presentation reading, but the pillar comparison is exported as a separate figure so it can be laid out cleanly in PowerPoint.

In [27]:
slide2_metrics = pd.DataFrame([
    {
        'label': 'Total Approved',
        'value_2026': q1_2026['total_approved_chf'].sum(),
        'value_2025': q1_2025['total_approved_chf'].sum(),
        'formatter': 'chf'
    },
    {
        'label': 'Allocations',
        'value_2026': len(q1_2026),
        'value_2025': len(q1_2025),
        'formatter': 'num'
    },
    {
        'label': 'Supported Operations',
        'value_2026': q1_2026['appeal_id'].nunique(),
        'value_2025': q1_2025['appeal_id'].nunique(),
        'formatter': 'num'
    },
    {
        'label': 'Targeted People',
        'value_2026': q1_2026['targeted_people'].sum(min_count=1),
        'value_2025': q1_2025['targeted_people'].sum(min_count=1),
        'formatter': 'num'
    },
    {
        'label': 'Countries',
        'value_2026': q1_2026['country'].nunique(),
        'value_2025': q1_2025['country'].nunique(),
        'formatter': 'num'
    }
])
slide2_metrics['delta'] = slide2_metrics['value_2026'] - slide2_metrics['value_2025']
slide2_metrics['delta_pct'] = np.where(
    slide2_metrics['value_2025'].fillna(0).eq(0),
    np.nan,
    slide2_metrics['delta'] / slide2_metrics['value_2025']
)

protocol_counts = protocol_context['appeal_type'].value_counts()
trigger_text = f"Triggered protocols context: {len(protocol_context)} total ({protocol_counts.get('EAP', 0)} EAP, {protocol_counts.get('s-EAP', 0)} s-EAP)"

slide2_metrics

,label,value_2026,value_2025,formatter,delta,delta_pct
0,Total Approved,"15,962,603.00","9,047,619.00",chf,"6,914,984.00",0.76
1,Allocations,54.00,33.00,num,21.00,0.64
2,Supported Operations,51.00,33.00,num,18.00,0.55
3,Targeted People,"1,833,048.00","4,810,331.00",num,"-2,977,283.00",-0.62
4,Countries,46.00,28.00,num,18.00,0.64


In [28]:
# ── Slide 2 KPI Cards — roomier layout with explicit 2025 baselines ──────────
fig = go.Figure()

card_sequence = ['Total Approved', 'Allocations', 'Countries', 'Supported Operations', 'Targeted People']
card_metrics = slide2_metrics.set_index('label').loc[card_sequence].reset_index()

domains = [
    {'x': [0.00, 0.30], 'y': [0.57, 1.00]},
    {'x': [0.35, 0.65], 'y': [0.57, 1.00]},
    {'x': [0.70, 1.00], 'y': [0.57, 1.00]},
    {'x': [0.00, 0.30], 'y': [0.00, 0.43]},
    {'x': [0.35, 0.65], 'y': [0.00, 0.43]},
]
note_domain = {'x': [0.70, 1.00], 'y': [0.00, 0.43]}


def indicator_spec(row):
    if row.label == 'Total Approved':
        return row.value_2026 / 1_000_000, row.value_2025 / 1_000_000, 'CHF ', 'M', '.1f'
    if row.label == 'Targeted People':
        return row.value_2026 / 1_000_000, row.value_2025 / 1_000_000, '', 'M', '.1f'
    return row.value_2026, row.value_2025, '', '', ',.0f'


for row, domain in zip(card_metrics.itertuples(index=False), domains):
    value, reference, prefix, suffix, valueformat = indicator_spec(row)
    baseline_text = human_chf(row.value_2025) if row.formatter == 'chf' else human_number(row.value_2025)

    fig.add_trace(
        go.Indicator(
            mode='number+delta',
            value=value,
            number={
                'prefix': prefix,
                'suffix': suffix,
                'valueformat': valueformat,
                'font': {'size': 46, 'color': INK, 'family': FLR_FONT},
            },
            delta={
                'reference': reference,
                'relative': True,
                'valueformat': '+.0%',
                'font': {'size': 18, 'family': FLR_FONT},
                'position': 'bottom',
                'increasing': {'color': '#2e8b57'},
                'decreasing': {'color': IFRC_RED},
            },
            title={
                'text': (
                    f'<b>{row.label}</b><br>'
                    f'<span style="font-size:14px;color:{FLR_MUTED};">'
                    f'2025 Q1 baseline: {baseline_text}'
                    '</span>'
                ),
                'font': {'size': 20, 'color': NAVY, 'family': FLR_FONT},
            },
            domain=domain,
        )
    )

for domain in domains + [note_domain]:
    fig.add_shape(
        type='rect',
        xref='paper',
        yref='paper',
        x0=domain['x'][0],
        x1=domain['x'][1],
        y0=domain['y'][0],
        y1=domain['y'][1],
        line={'color': FLR_LINE, 'width': 1.2},
        fillcolor='white',
        layer='below',
    )

fig.add_annotation(
    x=(note_domain['x'][0] + note_domain['x'][1]) / 2,
    y=(note_domain['y'][0] + note_domain['y'][1]) / 2,
    xref='paper',
    yref='paper',
    text=(
        '<b>Triggered protocols context</b><br>'
        f'<span style="font-size:18px;color:{NAVY};"><b>{len(protocol_context)} total</b></span><br>'
        f'<span style="font-size:15px;color:{FLR_MUTED};">'
        f"{protocol_counts.get('EAP', 0)} EAP and {protocol_counts.get('s-EAP', 0)} s-EAP"
        '</span>'
    ),
    showarrow=False,
    align='center',
    font={'size': 18, 'color': NAVY, 'family': FLR_FONT},
)

fig.update_layout(
    **flr_layout(
        'Slide 2: Headline KPI Cards',
        subtitle='DREF family Q1 2026 with explicit Q1 2025 baselines',
        height=980,
        margin_l=40,
        margin_r=40,
        margin_t=120,
        margin_b=40,
    ),
)

save_figure(fig, 'slide_02_kpi_cards', width=1400, height=980)
fig

In [37]:
# ── Slide 2 Pillar Share Comparison ─────────────────────────────────────────
pillar_compare = pd.concat([
    q1_2025.assign(period='2025 Q1'),
    q1_2026.assign(period='2026 Q1')
]).groupby(['period', 'pillar'], as_index=False)['total_approved_chf'].sum()

totals = pillar_compare.groupby('period', as_index=False)['total_approved_chf'].sum().rename(columns={'total_approved_chf': 'period_total'})
pillar_compare = pillar_compare.merge(totals, on='period', how='left')
pillar_compare['share'] = pillar_compare['total_approved_chf'] / pillar_compare['period_total']

total_2025 = float(totals.loc[totals['period'] == '2025 Q1', 'period_total'].iloc[0])
total_2026 = float(totals.loc[totals['period'] == '2026 Q1', 'period_total'].iloc[0])
yoy_chf_pct = (total_2026 - total_2025) / total_2025

fig = go.Figure()
for pillar, color in [('Response', IFRC_RED), ('Anticipatory', AMBER)]:
    subset = pillar_compare[pillar_compare['pillar'] == pillar]
    fig.add_trace(go.Bar(
        x=subset['share'],
        y=subset['period'],
        orientation='h',
        name=pillar,
        marker_color=color,
        marker_line_width=0,
        text=[f"{share:.1%}<br>{human_chf(value)}" for share, value in zip(subset['share'], subset['total_approved_chf'])],
        textposition='inside',
        textfont={'size': 16, 'color': 'white', 'family': FLR_FONT},
    ))

for row in totals.itertuples(index=False):
    fig.add_annotation(
        x=1.01,
        y=row.period,
        xref='x',
        yref='y',
        text=f'<b>{human_chf(row.period_total)}</b>',
        showarrow=False,
        xanchor='left',
        font={'size': 20, 'color': INK, 'family': FLR_FONT},
    )

# Title, subtitle, and insight caption all stacked tightly — no floating annotations
title_html = (
    'Pillar Share Comparison'
    f'<br><span style="font-size:17px;color:{FLR_MUTED};">Response vs Anticipatory - Q1 2025 vs Q1 2026</span>'
    f'<br><span style="font-size:14px;color:#555;">'
    f'Total Q1 allocations grew <b>{yoy_chf_pct:+.0%}</b> year-over-year, '
    'driven overwhelmingly by the Response pillar</span>'
)

fig.update_layout(
    barmode='stack',
    title={
        'text': title_html,
        'x': 0.5,
        'xanchor': 'center',
        'y': 0.9,
        'yanchor': 'top',
        'font': FLR_TITLE,
        'pad': {'t': 8, 'b': 20},
    },
    paper_bgcolor=FLR_PAPER,
    plot_bgcolor=FLR_PLOT,
    margin={'l': 95, 'r': 165, 't': 190, 'b': 80},
    height=600,
    font={'family': FLR_FONT, 'size': 15, 'color': '#444444'},
    hoverlabel=dict(bgcolor='white', bordercolor='#ddd', font_size=13, font_family=FLR_FONT),
    legend=flr_legend(y=1.05),
    xaxis=flr_xaxis('Share of Q1 approved CHF', tickformat='.0%', range=[0, 1.14]),
    yaxis=flr_yaxis('', showgrid=False, categoryorder='array', categoryarray=['2026 Q1', '2025 Q1']),
)

save_figure(fig, 'slide_02_pillar_share_compare', width=1400, height=600)
fig


## Slide 3

This slide now uses historical context small multiples. Each panel shows the 2022 to 2026 Q1 series, keeps 2025 as the immediate baseline, and highlights 2026 so the audience can see whether the latest value is a continuation or a step change.

In [38]:
# ── Slide 3: Q1 2026 in context — historical small multiples ─────────────────
slide3_yearly = (
    q1_history[q1_history['approval_year'].between(2022, 2026)]
    .groupby('approval_year')
    .agg(
        total_approved_chf=('total_approved_chf', 'sum'),
        allocations=('appeal_id', 'count'),
        supported_operations=('appeal_id', 'nunique'),
        targeted_people=('targeted_people', lambda values: values.sum(min_count=1)),
    )
    .reset_index()
)

slide3_specs = [
    {'column': 'total_approved_chf', 'label': 'Total Approved CHF', 'formatter': 'chf', 'axis_title': 'Approved CHF'},
    {'column': 'allocations', 'label': 'Allocations', 'formatter': 'count', 'axis_title': 'Allocations'},
    {'column': 'supported_operations', 'label': 'Supported Operations', 'formatter': 'count', 'axis_title': 'Operations'},
    {'column': 'targeted_people', 'label': 'Targeted People', 'formatter': 'people', 'axis_title': 'People'},
]


def slide3_metric_text(value, formatter):
    if formatter == 'chf':
        return human_chf(value)
    return human_number(value)


fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[spec['label'] for spec in slide3_specs],
    horizontal_spacing=0.14,
    vertical_spacing=0.24,
)

for index, spec in enumerate(slide3_specs, start=1):
    row = 1 if index <= 2 else 2
    col = 1 if index in [1, 3] else 2

    series = slide3_yearly[['approval_year', spec['column']]].copy()
    avg_2022_2024 = series.loc[series['approval_year'].between(2022, 2024), spec['column']].mean()
    value_2025 = float(series.loc[series['approval_year'].eq(2025), spec['column']].iloc[0])
    value_2026 = float(series.loc[series['approval_year'].eq(2026), spec['column']].iloc[0])
    delta_pct = np.nan if value_2025 == 0 else (value_2026 - value_2025) / value_2025
    delta_text = 'n/a' if np.isnan(delta_pct) else f'{delta_pct:+.0%} vs 2025'
    delta_color = '#2e8b57' if (not np.isnan(delta_pct) and delta_pct >= 0) else IFRC_RED

    fig.add_trace(
        go.Scatter(
            x=series['approval_year'],
            y=series[spec['column']],
            mode='lines+markers',
            line={'color': '#BCC5D3', 'width': 3},
            marker={
                'size': 10,
                'color': '#BCC5D3',
                'line': {'color': FLR_PAPER, 'width': 1.5},
            },
            showlegend=False,
            hoverinfo='skip',
        ),
        row=row,
        col=col,
    )

    fig.add_trace(
        go.Scatter(
            x=[2025],
            y=[value_2025],
            mode='markers',
            marker={'size': 16, 'color': NAVY, 'line': {'color': FLR_PAPER, 'width': 1.5}},
            showlegend=False,
            hoverinfo='skip',
        ),
        row=row,
        col=col,
    )

    fig.add_trace(
        go.Scatter(
            x=[2026],
            y=[value_2026],
            mode='markers',
            marker={'size': 18, 'color': IFRC_RED, 'line': {'color': FLR_PAPER, 'width': 1.5}},
            showlegend=False,
            hoverinfo='skip',
        ),
        row=row,
        col=col,
    )

    fig.add_hline(
        y=avg_2022_2024,
        line_dash='dot',
        line_color='#9AA3B2',
        line_width=2,
        annotation_text='2022-24 avg',
        annotation_position='top left',
        annotation_font={'size': 13, 'color': FLR_MUTED, 'family': FLR_FONT},
        row=row,
        col=col,
    )

    fig.add_annotation(
        x=2025,
        y=value_2025,
        text=slide3_metric_text(value_2025, spec['formatter']),
        showarrow=False,
        yshift=20,
        font={'size': 14, 'color': NAVY, 'family': FLR_FONT},
        row=row,
        col=col,
    )

    fig.add_annotation(
        x=2026,
        y=value_2026,
        text=(
            f'<b>{slide3_metric_text(value_2026, spec["formatter"])}</b><br>'
            f'<span style="color:{delta_color};">{delta_text}</span>'
        ),
        showarrow=False,
        xanchor='left',
        xshift=10,
        yshift=5,
        font={'size': 15, 'color': INK, 'family': FLR_FONT},
        row=row,
        col=col,
    )

    axis_overrides = {}
    if spec['formatter'] == 'chf':
        axis_overrides = {'tickformat': '~s', 'tickprefix': 'CHF '}
    elif spec['formatter'] == 'people':
        axis_overrides = {'tickformat': '~s'}
    else:
        axis_overrides = {'tickformat': ',d'}

    fig.update_xaxes(
        **flr_xaxis('Q1 approval year' if row == 2 else ''),
        tickmode='array',
        tickvals=[2022, 2023, 2024, 2025, 2026],
        range=[2021.75, 2026.35],
        row=row,
        col=col,
    )
    fig.update_yaxes(
        **flr_yaxis(spec['axis_title'], range=[0, series[spec['column']].max() * 1.35], **axis_overrides),
        row=row,
        col=col,
    )

for annotation in fig.layout.annotations:
    if annotation.text in [spec['label'] for spec in slide3_specs]:
        annotation.update(font={'size': 18, 'color': '#444', 'family': FLR_FONT})

fig.update_layout(
    **flr_layout(
        'Q1 2026 In Context',
        subtitle='2025 shown as the baseline year; dotted lines mark the 2022-2024 average',
        height=980,
        margin_l=85,
        margin_r=85,
        margin_t=185,
        margin_b=85,
    ),
)

# Append insight as a 3rd title line — sits below subtitle, above the charts
fig.layout.title.text += (
    f'<br><span style="font-size:14px;color:#555;">'
    '2026 reached a new Q1 high for approved CHF, allocations and supported operations, '
    'but targeted people remained well below the 2023 and 2025 peaks.</span>'
)

save_figure(fig, 'slide_03_q1_comparison_panels', width=1400, height=980)
fig


## Slide 4

This figure remains provisional because `crisis_categorization` is sparse in the base data. The redesign keeps a single readable stacked bar chart, removes the redundant lower panel, and makes the coverage caveat explicit.

In [50]:
# ── Slide 4: Crisis Category Mix — single-panel provisional view ─────────────
crisis = q1_history[q1_history['crisis_categorization'].notna() & q1_history['crisis_categorization'].ne('')].copy()
crisis = crisis[crisis['approval_year'].between(2022, 2026)]
crisis_mix = crisis.groupby(['approval_year', 'crisis_categorization'], as_index=False)['total_approved_chf'].sum()
crisis_mix['year_label'] = 'Q1 ' + crisis_mix['approval_year'].astype(str)
year_order = sorted(crisis_mix['year_label'].unique(), key=lambda value: int(value.split()[-1]))
year_totals = crisis_mix.groupby('year_label', as_index=False)['total_approved_chf'].sum()
base_years = pd.DataFrame({'year_label': year_order})

colors = {'Yellow': '#F5D98E', 'Orange': WARM_ORANGE, 'Red': IFRC_RED}

fig = go.Figure()

for category in ['Yellow', 'Orange', 'Red']:
    subset = base_years.merge(
        crisis_mix.loc[crisis_mix['crisis_categorization'] == category, ['year_label', 'total_approved_chf']],
        on='year_label',
        how='left',
    ).fillna({'total_approved_chf': 0})
    text_labels = [human_chf(value) if value >= 1_000_000 else '' for value in subset['total_approved_chf']]

    fig.add_trace(go.Bar(
        x=subset['year_label'],
        y=subset['total_approved_chf'],
        name=category,
        marker_color=colors[category],
        marker_line_width=0,
        text=text_labels,
        textposition='inside',
        textfont={'size': 14, 'color': 'white', 'family': FLR_FONT},
    ))

for row in year_totals.itertuples(index=False):
    fig.add_annotation(
        x=row.year_label,
        y=row.total_approved_chf,
        text=f'<b>{human_chf(row.total_approved_chf)}</b>',
        showarrow=False,
        yshift=22,
        font={'size': 16, 'color': INK, 'family': FLR_FONT},
    )
    
fig.update_layout(
    barmode='stack',
    **flr_layout(
        'Crisis Category Mix (Provisional)',
        subtitle='Single-panel view of categorized rows only; use with workbook reconciliation caveat',
        height=750,
        margin_l=100,
        margin_r=20,
        margin_t=200,
        margin_b=40,
    ),
    legend=flr_legend(y=1.2),
)

# Insight caption — placed above bars (in top margin), below legend
fig.add_annotation(
    x=0.5,
    y=1.10,
    xref='paper',
    yref='paper',
    text='Within the categorized subset, Yellow remains dominant, but Orange rose from 11% in 2025 to 23% in 2026.',
    showarrow=False,
    font={'size': 15, 'color': '#555', 'family': FLR_FONT},
)


fig.update_yaxes(**flr_yaxis('Approved CHF', tickformat='~s', tickprefix='CHF '))
fig.update_xaxes(**flr_xaxis('Approval year', type='category', categoryorder='array', categoryarray=year_order))

save_figure(fig, 'slide_04_crisis_mix_provisional', width=1400, height=720)
fig


## Slide 5

The localization slide is built as a bullet-style benchmark figure. The values currently come from the published slide because no explicit localization field is present in `ALL_DATA`.

In [32]:
# ── Slide 5: Localization Benchmark with gap-to-target ───────────────────────
localization = pd.DataFrame({
    'label': ['Two Pillars', 'Anticipatory Pillar', 'Response Pillar'],
    'value': [79.5, 86.0, 73.0],
    'target': [80.0, 80.0, 80.0]
})
localization = localization.iloc[::-1].reset_index(drop=True)
localization['gap'] = localization['value'] - localization['target']

PILLAR_BAR_COLORS = [IFRC_RED, AMBER, NAVY]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=[100] * len(localization),
    y=localization['label'],
    orientation='h',
    marker_color='#EEEEEE',
    showlegend=False,
    hoverinfo='skip',
))
fig.add_trace(go.Bar(
    x=localization['value'],
    y=localization['label'],
    orientation='h',
    marker_color=PILLAR_BAR_COLORS,
    marker_line_width=0,
    showlegend=False,
    text=[f'{value:.1f}%' for value in localization['value']],
    textposition='inside',
    textfont={'size': 17, 'color': 'white', 'family': FLR_FONT},
))

for i, row in enumerate(localization.itertuples(index=False)):
    fig.add_shape(
        type='line',
        x0=row.target,
        x1=row.target,
        y0=i - 0.38,
        y1=i + 0.38,
        xref='x',
        yref='y',
        line={'color': INK, 'width': 3, 'dash': 'dot'},
    )

    gap_color = '#2a7f2a' if row.gap >= 0 else IFRC_RED
    gap_sign = '+' if row.gap >= 0 else ''
    gap_symbol = '✓' if row.gap >= 0 else '▼'
    fig.add_annotation(
        x=max(row.value, row.target) + 2.2,
        y=row.label,
        text=f'<b>{gap_symbol} {gap_sign}{row.gap:.1f}pp</b>',
        showarrow=False,
        xanchor='left',
        font={'size': 16, 'color': gap_color, 'family': FLR_FONT},
    )

fig.add_annotation(
    x=80,
    y=1.08,
    xref='x',
    yref='paper',
    text='80% target',
    showarrow=False,
    font={'size': 14, 'color': '#666', 'family': FLR_FONT},
)

fig.update_layout(
    barmode='overlay',
    **flr_layout(
        'Localization Benchmark',
        subtitle='Provisional - localization field not reproducible from raw data',
        height=500,
        margin_l=175,
        margin_r=115,
        margin_t=125,
        margin_b=75,
    ),
    xaxis=flr_xaxis('Localization share', range=[0, 105], ticksuffix='%'),
    yaxis=flr_yaxis('', showgrid=False),
)

save_figure(fig, 'slide_05_localization_bullets', width=1400, height=500)
fig